[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/klar74/WS2026_lecture/blob/main/Vorlesung_26/VL26_Word_Embeddings.ipynb)

# VL26 Notebook 1: Word Embeddings
## Von One-Hot zu Word2Vec

In diesem Notebook lernen wir:
- Wie One-Hot Encoding funktioniert (und warum es nicht ausreicht)
- Wie Word2Vec aus Wortumgebungen Vektoren lernt
- Wie Wortbeziehungen manchmal als Vektor-Rechnung sichtbar werden (z. B. König - Mann + Frau ≈ Königin)
- Die Limitation: Ein Wort = Ein Vektor (kein Kontext)

In [ ]:
# Installation (nur einmal ausführen)
# !pip install gensim numpy matplotlib scikit-learn

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE
from gensim.models import Word2Vec

## 1. One-Hot Encoding: Die naive Lösung

Jedes Wort bekommt eine Position in einem Vektor. Nur diese Position ist 1, alle anderen 0.

In [ ]:
# Beispiel-Vokabular
vocab = ["Hund", "Katze", "Vogel", "Auto"]

# One-Hot Vektoren erstellen
def one_hot(word, vocab):
    vec = np.zeros(len(vocab))
    vec[vocab.index(word)] = 1
    return vec

# Vektoren ausgeben
for word in vocab:
    print(f"{word:10s}: {one_hot(word, vocab)}")

**Problem:** Alle Wörter haben gleichen Abstand zueinander!
- Abstand Hund-Katze = Abstand Hund-Auto
- Keine Bedeutung kodiert

## 2. Word2Vec: Eigenes Training auf Beispiel-Korpus

**Realität:** Word2Vec braucht normalerweise VIELE Daten (Millionen Sätze aus Wikipedia, Nachrichten, etc.).

**Unser Ansatz:** Wir trainieren ein kleines Modell auf 32 selbst zusammengestellten Sätzen. Dieses Beispiel ist unabhängig von der One-Hot-Liste oben: Das Word2Vec-Vokabular besteht aus allen Wörtern der Trainingssätze, nicht nur aus `Hund`, `Katze`, `Vogel` und `Auto`.

Jeder Satz ist eine Liste von Wörtern. Bei Skip-Gram (`sg=1`) lernt das Modell, welche Wörter in der Nähe eines Wortes vorkommen. Zum Beispiel steht `weise` im Satz `eine weise königin entscheidet` und gehört deshalb zum Vokabular. `weise` und `weiser` gelten als unterschiedliche Wörter.

Die 32 Sätze stehen jeweils einmal im Korpus. Das Modell verarbeitet sie in mehreren **Epochen**: `epochs=1000` bedeutet 1.000 Trainingsdurchläufe. Weitere Durchläufe verändern die Vektoren, liefern aber keine neuen Sätze oder Kontexte; das kleine Beispiel bleibt künstlich und kann unzuverlässige Ergebnisse liefern.

**Wichtig:** Für aussagekräftige Wortvektoren benötigt man viel vielfältigere Texte oder ein geeignetes vortrainiertes Modell.

In [ ]:
# Eigenen deutschen Trainingskorpus erstellen
# In der Praxis würde man auf Wikipedia, Nachrichten, Büchern trainieren

# Deutscher Trainingskorpus mit vielen Beispielsätzen
base_sentences = [
    # Tiere
    ["der", "hund", "bellt", "laut"],
    ["die", "katze", "miaut", "leise"],
    ["der", "hund", "jagt", "die", "katze"],
    ["der", "vogel", "singt", "schön"],
    ["der", "hund", "ist", "ein", "tier"],
    ["die", "katze", "ist", "ein", "tier"],
    
    # Fahrzeuge
    ["das", "auto", "fährt", "schnell"],
    ["das", "fahrrad", "ist", "langsam"],
    
    # König/Königin - Mann/Frau
    ["der", "könig", "regiert", "das", "land"],
    ["die", "königin", "regiert", "das", "land"],
    ["könig", "und", "königin", "sind", "verheiratet"],
    ["der", "könig", "ist", "ein", "mann"],
    ["die", "königin", "ist", "eine", "frau"],
    ["ein", "mann", "arbeitet", "hart"],
    ["eine", "frau", "arbeitet", "hart"],
    ["der", "mann", "geht", "spazieren"],
    ["die", "frau", "geht", "spazieren"],
    ["jeder", "mann", "hat", "rechte"],
    ["jede", "frau", "hat", "rechte"],
    
    # Weitere Paare
    ["der", "vater", "ist", "ein", "mann"],
    ["die", "mutter", "ist", "eine", "frau"],
    ["vater", "und", "mutter", "sind", "eltern"],
    ["der", "bruder", "ist", "männlich"],
    ["die", "schwester", "ist", "weiblich"],
    ["mein", "bruder", "hilft", "mir"],
    ["meine", "schwester", "hilft", "mir"],
    ["der", "onkel", "ist", "verwandt"],
    ["die", "tante", "ist", "verwandt"],
    
    # Adjektive für König/Königin
    ["der", "mächtige", "könig", "herrscht"],
    ["die", "mächtige", "königin", "herrscht"],
    ["ein", "weiser", "könig", "entscheidet"],
    ["eine", "weise", "königin", "entscheidet"],
]

# Jeder Satz steht nur einmal im Korpus; epochs steuert die Trainingsdurchläufe.
sentences = base_sentences

print(f"Trainingskorpus erstellt: {len(sentences)} Sätze")
print("Starte Training...")

# Kleines Demonstrationsmodell trainieren
model = Word2Vec(
    sentences, 
    vector_size=150,
    window=7,
    min_count=1, 
    sg=1,                 # Skip-Gram
    epochs=1000,          # 1.000 Durchläufe über dieselben 32 Sätze
    negative=10,          # Negative Sampling
    alpha=0.025,          # Anfangslernrate
    min_alpha=0.0001,     # Lernrate am Ende
    seed=42,
    workers=1,
)

print("✓ Modell trainiert!")
print(f"  Vokabular-Größe: {len(model.wv)} Wörter")
print(f"  Trainings-Sätze: {len(sentences)}")
print(f"  Trainingsdurchläufe: {model.epochs}")
print(f"  Dimensionen: {model.vector_size}")

## 3. Beispiel für Vektor-Arithmetik

`model.wv` enthält die gelernten Wortvektoren. Beispielsweise liefert `model.wv['könig']` den Vektor mit 150 Zahlen. Das Wort muss im Trainingskorpus vorkommen.

`most_similar('könig', topn=5)` sucht die fünf Wörter mit den ähnlichsten Vektoren. Dafür verwendet es die **Kosinus-Ähnlichkeit**: Sie vergleicht die Richtung der Vektoren; je größer der Wert, desto ähnlicher die Richtung.

Bei der bekannten Analogie `König - Mann + Frau` kombiniert `most_similar` die Richtungen der drei Wortvektoren und sucht die nächstliegenden Wörter. Die eingegebenen Wörter selbst werden nicht als Treffer ausgegeben. In manchen größeren Modellen liegt `königin` dabei weit oben – garantiert ist das nicht.

**Warum erscheint manchmal `weise`?** Das Wort steht im Trainingssatz `eine weise königin entscheidet`. Es ist daher ein möglicher Treffer, ebenso wie andere Wörter des Korpus. Ein Treffer bedeutet nicht, dass die Wörter dieselbe Bedeutung haben. Die Ähnlichkeitswerte sind keine Wahrscheinlichkeiten. Sie hängen vom Korpus und vom Training ab; besonders bei 32 Sätzen sind sie nicht zuverlässig.

In [ ]:
result = model.wv.most_similar(
    positive=['könig', 'frau'],
    negative=['mann'],
    topn=5,
)

print("Treffer für König - Mann + Frau:")
for word, score in result:
    print(f"  {word:15s} (Kosinus-Ähnlichkeit: {score:.3f})")

# Zeigt, woher das Wort im Trainingskorpus stammt.
print("\nTrainingssätze mit 'weise':")
for sentence in sentences:
    if 'weise' in sentence:
        print(" ", ' '.join(sentence))

print("\nWeitere Analogien zum Ausprobieren:")
for word in ['vater', 'bruder']:
    result = model.wv.most_similar(positive=[word, 'frau'], negative=['mann'], topn=3)
    print(f"  {word} - mann + frau: {result}")

## 4. Visualisierung: Wörter im 2D-Raum

Wir reduzieren die 150 Dimensionen mit t-SNE auf zwei Dimensionen und zeichnen die Wörter. Dabei gehen Informationen verloren; die Abstände in der Grafik entsprechen nicht unbedingt den Kosinus-Ähnlichkeiten der ursprünglichen Vektoren.

In [ ]:
# Alle Wörter und ihre Vektoren extrahieren
words = list(model.wv.index_to_key)
vectors = np.array([model.wv[word] for word in words])

# t-SNE: Dimensionsreduktion auf 2D
tsne = TSNE(n_components=2, random_state=42, perplexity=min(10, len(words)-1))
vectors_2d = tsne.fit_transform(vectors)

# Plot erstellen
plt.figure(figsize=(14, 10))
plt.scatter(vectors_2d[:, 0], vectors_2d[:, 1], alpha=0.5, s=100)

# Wichtige Wörter hervorheben
wichtige_worte = ['könig', 'königin', 'mann', 'frau', 'vater', 'mutter', 'bruder', 'schwester']
for i, word in enumerate(words):
    if word in wichtige_worte:
        plt.annotate(word, (vectors_2d[i, 0], vectors_2d[i, 1]), 
                    fontsize=12, fontweight='bold', alpha=0.9,
                    bbox=dict(boxstyle='round,pad=0.3', facecolor='yellow', alpha=0.7))
    else:
        plt.annotate(word, (vectors_2d[i, 0], vectors_2d[i, 1]), 
                    fontsize=9, alpha=0.6)

plt.title("Word2Vec Embeddings (2D Projektion)", fontsize=14)
plt.xlabel("Dimension 1")
plt.ylabel("Dimension 2")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

**Beobachtung:**
- Prüfe, ob ähnliche Wörter (z. B. Hund und Katze) in der Grafik nahe beieinander liegen.
- t-SNE erstellt nur eine 2D-Ansicht. Die Grafik ist ein Hinweis, aber kein Beweis für gelernte Bedeutungen.

## 5. Die fundamentale Limitation: Ein Wort = Ein Vektor

**Problem:** Word2Vec gibt jeder Wortform EINEN festen Vektor, egal in welchem Kontext sie verwendet wird. Die beiden Sätze unten enthalten `bank` einmal als Geldinstitut und einmal als Sitzmöbel. Beide Verwendungen fließen in denselben Vektor ein.

In [ ]:
# Beispiel: "Bank" hat IMMER denselben Vektor
# Auch wenn wir zusätzliche Sätze mit unterschiedlichen Bedeutungen hätten:

bank_sentences = [
    ["ich", "gehe", "zur", "bank", "und", "hole", "geld"],  # Geldinstitut
    ["ich", "sitze", "auf", "der", "bank", "im", "park"],   # Sitzmöbel
]

# Neues Modell mit Bank-Beispielen
extended_sentences = sentences + bank_sentences
model_bank = Word2Vec(extended_sentences, vector_size=150, window=7, min_count=1, sg=1, epochs=500, seed=42, workers=1)

# "bank" hat nur EINEN Vektor, unabhängig vom Satz.
print("Vektor für 'bank' (erste 10 Dimensionen):")
print(model_bank.wv["bank"][:10])
print("\nDerselbe Vektor wird für beide Sätze verwendet.")

## 🎯 Takeaway

**Word2Vec Vorteile:**
- ✅ Ähnliche Wortumgebungen können zu ähnlichen Vektoren führen
- ✅ Vektor-Rechnungen können Beziehungen zeigen; mit unserem kleinen Korpus ist das Ergebnis nicht garantiert
- ✅ Die Ähnlichkeit gelernter Vektoren lässt sich vergleichen

**Word2Vec Limitation:**
- ❌ Ein Wort = Ein Vektor (kontextunabhängig)
- ❌ "Bank" hat denselben Vektor bei "Geld holen" und "im Park sitzen"
- ❌ Verschiedene Bedeutungen eines Wortes erhalten keine eigenen kontextabhängigen Vektoren

**Lösung:** Wir brauchen **kontextabhängige Embeddings** → Das führt uns zu BERT (Notebook 2)!